<a href="https://colab.research.google.com/github/DavidSenseman/STA1403/blob/main/STA1403_Chapter_09A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- STA1403_CHAPTER_09A:Rev 1 -->

---------------------------
**COPYRIGHT NOTICE:** This Jupyterlab Notebook is a Derivative work of [Jeff Heaton](https://github.com/jeffheaton) licensed under the Apache License, Version 2.0 (the "License"); You may not use this file except in compliance with the License. You may obtain a copy of the License at

> [http://www.apache.org/licenses/LICENSE-2.0](http://www.apache.org/licenses/LICENSE-2.0)

Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.

------------------------

# **STA 1403: Biostatistics**

## **Analysis of Variance (ANOVA)**

* Instructor: [David Senseman](mailto:David.Senseman@utsa.edu), [Department of Biology, Health and the Environment](https://sciences.utsa.edu/bhe/), [UT San Antonio](https://www.utsa.edu/)

##### Contents
* **9.1 : Introduction**
* **9.2 : Analysis of Variance (ANOVA)**
* 9.3 : The Assumptions of ANOVA
* 9.4 : Advanced

## Google Colab Instructions

Run next code cell to map this Colab lesson's folder /content/drive to your Google Drive. This will allow you keep a copy of this Colab notebook which **_is_** your course textbook.

In [ ]:
# @title **Run this cell first**
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    Colab = True
    print("Note: Using Google CoLab")
    !curl ipinfo.io
except:
    print("**WARNING**: Your GDrive is not mapped to /content/drive ")
    print("**WARNING**: Your work will not be saved!")
    Colab = False

You should see something _similar_ to the following output:
```text
Mounted at /content/drive
Note: Using Google CoLab
{
  "ip": "34.139.115.187",
  "hostname": "187.115.139.34.bc.googleusercontent.com",
  "city": "North Charleston",
  "region": "South Carolina",
  "country": "US",
  "loc": "32.8546,-79.9748",
  "org": "AS396982 Google LLC",
  "postal": "29415",
  "timezone": "America/New_York",
  "readme": "https://ipinfo.io/missingauth"
}
```

## Test Your STA1403 Key

Run the next cell to test whether your STA1403 Key is correctly installed in your Colab Secrets.

In [ ]:
# @title Test Your STA1403 Key

from google.colab import userdata
import os

# Check if STA1403 key is properly loaded
try:
    # 1. Get the key from Secrets
    STA1403_KEY = userdata.get('STA1403_KEY')

    # 2. Set it as an environment variable
    os.environ['STA1403_KEY'] = STA1403_KEY

    print("STA1403 key loaded and environment variable set successfully!")
    print(f"Key length: {len(STA1403_KEY)}")

except Exception as e:
    print(f"Error loading STA1403 key: {e}")
    print("Please set your STA1403 key in Google Colab:")
    print("1. Go to Secrets in the left sidebar (key icon)")
    print("2. Create a new secret named 'STA1403_KEY'")
    print("3. Paste your STA1403 key and toggle 'Notebook access' on")

If your STA1403 key is correctly stored in your Colab Secrets, you should see the following output:
```text
STA1403 key loaded and environment variable set successfully!
Key length: 28
```
Contact your Instructor if you recieve an error since you won't be able to submit your lesson for grading unless your STA1403 key is correctly installed.

## Load Data Set

Run the next cell to load the data set for this lesson.

In [ ]:
# @title Load Data Set
import pandas as pd

# Load dataset
df_cushings = pd.read_csv("https://biologicslab.co/STA1403/data/Cushings.csv",
                         sep = ',', na_values=[" ", "NA", "null", ""])

print("Data set loaded sucessfully. ✅")

# **Chapter 9 : Analysis of Variance (ANOVA)**

## **9.1 Introduction**

In Chap. 8, we discussed how two-sample $t$-tests can be used to evaluate hypotheses regarding the difference between the means of two groups. We mentioned that we typically use this approach to investigate the relationship between a binary categorical (factor) variable, which specifies the two groups, and a numerical variable, which is regarded as the response variable. In this chapter, we discuss Analysis of Variance (ANOVA) models that generalize the $t$-test and are used to compare the means of multiple groups identified by a categorical variable with more than two possible categories. As before, the categorical variable is called the **factor** and is typically considered as the explanatory variable. In contrast, the numerical variable, whose means across different groups are compared, is regarded as the response variable.

In this chapter, we mainly focus on ANOVA models with only one factor. These models are known as **one-way ANOVA**. In Advanced section, we briefly discuss **two-way ANOVA** models that include two factors (i.e., two categorical explanatory variables) in the analysis.

## **9.2 Analysis of Variance (ANOVA)**

As an example, we analyze the `Cushings` data set. Cushing's syndrome is a hormone disorder associated with high level of cortisol secreted by the adrenal gland. The Cushing's data set includes 27 observations $(n = 27)$. For each individual in the sample, the urinary excretion rates of two steroid metabolites are recorded. These are urinary excretion rate (mg/24 hr) of `Tetrahydrocortisone` and urinary excretion rate (mg/24 hr) of `Pregnanetriol`. The `Type` variable in the data set shows the underlying type of syndrome, which can be one of four categories: adenoma (a), bilateral hyperplasia (b), carcinoma (c), and unknown (u). Run the next cell to see the first 5 records in the `Cushings` dataset.

In [ ]:
# @title **Fig. 9.1 Cushings Data Set**
df_cushings.head()

**Fig. 9.1** Viewing the `Cushings` data set stored in the Pandas DataFrame `df_cushings`. For 27 individuals, the urinary
excretion rates of two steroid  metabolites and the underlying type of syndrome are recorded. The column name `TCort` is an abbreviation for the urinary metabolite `Tetrahydrocortisone` while the column name `PregN`, is an abbreviation for the urinary metabolite `Pregnanetriol`.

Our objective is to find whether the four groups are different with respect to urinary excretion rate of these two metabolites. We denote by $Y$ the urinary excretion rate of the metabolite and by $X$ the `Type` variable, where $X = 1$ for `Type = a`, $X = 2$ for `Type = b`, $X = 3$ for `Type = c`, and $X = 4$ for `Type = u`. Then, our objective could be defined as investigating whether the mean of the response variable $Y$ differs for different values (levels) of the factor $X$.

### _Test Data for Outliers_

Before we begin any analysis, it is a good idea to examine the reliability of the data set. For example, is there any reason for us to question whether one (or more) data values were correctly recorded? A common method to detect for "bad data" is to **test for outliers**. An outlier is a data point that differs significantly from the other observations in a dataset. It is a value that lies an abnormal distance from the rest of the data, often appearing as an extreme minimum or maximum.

In statistical terms, outliers are often identified using methods like the Z-score (e.g., more than 3 standard deviations from the mean) or the Interquartile Range (IQR), i.e., values falling more than 1.5 times the IQR below the first quartile or above the third quartile.

Example 1 shows how to use Python to test for potential outlier values.

## Example 1: Test for Outliers

The code in the cell below tests for possible outlier values for the `Pregnanetriol` measurements in the `df_cushings` DataFrame. The code uses the factor's **Interquartile Range (IQR)** as a baseline to set a `lower bound` and a `upper bound` to test for data values that are too small, or too large, respectively.  

In [ ]:
# @title Example 1: Test for Outliers

import pandas as pd
import numpy as np

# ── INPUT SECTION ─────────────────────────────────────────────────────────────
anova_factor   = "PregN"
factor_name    = "Pregnanetriol"
# ──────────────────────────────────────────────────────────────────────────────

# Create new DataFrame containing only one factor
anova_df = df_cushings[[anova_factor]].copy()

# 1. Calculate Quartiles
Q1 = anova_df[anova_factor].quantile(0.25)
Q3 = anova_df[anova_factor].quantile(0.75)
IQR = Q3 - Q1

# 2. Define Bounds
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 3. Flag Outliers
# Create a boolean column where True indicates an outlier
anova_df['is_outlier'] = (anova_df[anova_factor] < lower_bound) | (anova_df[anova_factor] > upper_bound)

# 4. Print results
# print(f"Lower Bound: {lower_bound}")
# print(f"Upper Bound: {upper_bound}")
print(f"\nPossible {factor_name} Outliers:")
print(anova_df)

If the code is correct, you should see the following output:
```text
Possible Pregnanetriol Outliers:
    PregN  is_outlier
0   11.70        True
1    1.30       False
2    0.10       False
3    0.04       False
4    1.10       False
5    0.40       False
6    1.00       False
7    0.20       False
8    0.60       False
9    1.20       False
10   0.60       False
11   3.60       False
12   1.60       False
13   0.40       False
14   0.40       False
15   1.60       False
16   6.40        True
17   7.90        True
18   3.10       False
19   2.50       False
20   7.60        True
21   0.40       False
22   5.00       False
23   0.80       False
24   0.10       False
25   0.10       False
26   0.80       False

```

Our test found 4 _possible_ outliers for the `Pregnanetriol` measurements, sample numbers `0`, `16` and `20`.

## **Exercise 1: Test for Outliers**

In the cell below, write the code to test for extreme values in the `Tetrahydrocortisone` measurements.

**Code Hints:**

1. Copy-and-paste Example 1 into the cell below
2. Change the variables in the INPUT SECTION to read as shown in the following code chunk:

```Python

# ── INPUT SECTION ─────────────────────────────────────────────────────────────
anova_factor  = 'TCort'
factor_name   = 'Tetrahydrocortisone'
# ──────────────────────────────────────────────────────────────────────────────
```
   

In [ ]:
# @title Exercise 1: Test for Outliers



If the code is correct, you should see the following output:
```text
Possible Tetrahydrocortisone Outliers:
    TCort  is_outlier
0     3.1       False
1     3.0       False
2     1.9       False
3     3.8       False
4     4.1       False
5     1.9       False
6     8.3       False
7     3.8       False
8     3.9       False
9     7.8       False
10    9.1       False
11   15.4       False
12    7.7       False
13    6.5       False
14    5.7       False
15   13.6       False
16   10.2       False
17    9.2       False
18    9.6       False
19   53.8        True
20   15.8       False
21    5.1       False
22   12.9       False
23   13.0       False
24    2.6       False
25   30.0        True
26   20.5       False
```

The output above shows two records `19` and `25` that are possible outliers for the metabolite `Tetrahydrocortisone`. Specifically, record `19` (`TCort` = $53.8$) and record `25` (`TCort` = $30.0$). Both samples are much higher than typical values observed for the variable `Tetrahydrocortisone`. We should further investigate this observation and remove it **_only_** if we are convinced that it was recorded by mistake and we cannot recover the correct values.

In what follows, we assume that this is a legitimate observation, so we include it in our analysis.

We start by focusing on `Tetrahydrocortisone`. We denote by $Y$ the urinary excretion rate of `Tetrahydrocortisone` and by $X$ the `Type` variable, where $X = 1$ for `Type = a`, $X = 2$ for `Type = b`, $X = 3$ for `Type = c`, and $X = 4$ for `Type = u`. Our objective is to investigate whether the _mean_ of the response variable $Y$ differs for different values (levels) of the factor $X$.

Denote the individual observations as $y_{ij}$: the urinary excretion rate of `Tetrahydrocortisone` of the $j$th individual in group $i$. The total number of observations is $n = 27$, and the number of observations in each group is $ n_{1} = 6$, $n_{2} = 10$, $n_{3} = 5$, and $n_{4} = 6$. The overall (for all groups) observed sample mean for the response variable is $\bar{y} = 10.46$. We also find the group specific means, which are $\bar{y}_{1} = 3.0$, $\bar{y}_{2} = 8.2$, $\bar{y}_{3} = 19.7$, and $\bar{y}_{4} = 14.0$.

![__](https://biologicslab.co/STA1403/images/chapter_09/chapter_09A_image01A.png)

>**Fig. 9.2** Strip chart of `Tetrahydrocortisone` by syndrome type. The overall sample mean $\bar{y}$ of `Tetrahydrocortisone` for all groups is shown as the _dashed horizontal line_, and the sample average $\bar{y}_{i}$ for each group is shown as the _small horizontal line_

Now, consider the dot plot of $Y$ by $X$ in Fig. 9.2. Here, each observation is represented by a point, and the overall average $\bar{y}$ of the response variable is represented by the dashed horizontal line at 10.46. Likewise, the sample average $(\bar{y}{1}, \ldots, \bar{y}{4})$ for each group is shown as a small horizontal line.

Across the four groups, there appears to be considerable variation in the group means (i.e., deviations of the small solid lines from the dashed line). Likewise, within groups, there are different degrees of variation of the observations from their specific mean (i.e., variation of points around the corresponding small horizontal line). Both sources of variation contribute to the total variation of the observations around the overall mean (dashed line).

------------------

>In general, the **between-groups variation** is denoted as $SS_{B}$ and calculated by
$$
SS_{B} = \sum_{i = 1}^ {k}n_{i} \left(\bar{y}_{i} - \bar {y}\right)^{2},  \tag{1}
$$
where $k$ is the number of groups (here, 4).

---------------

To find $SS_{B}$, we first find the squared difference between each group mean (i.e., the solid short lines) and the overall mean (i.e., the dashed line). In order to account for varying sample sizes, the squared distance is then multiplied by the number of observations in that group, $n_{i}$. (Therefore, groups with more observations are weighted more heavily.) The sum of these squared and weighted differences is the between-groups variation.

--------------

>The **within-groups variation** is denoted as $SS_{W}$ and calculated by
$$
SS_{W} = \sum_{i = 1}^{k} \sum_{j = 1}^{n_{i}} \left(y_{i j} - \bar{y}_{i}\right)^{2}.
$$

------------

To find $SS_{W}$, we first calculate the sum of squared deviations of each observation (i.e., the point) from the group mean (i.e., the short horizontal line) for each group separately. Then we add the results over all groups.

----------------

>We measure the **total variation** in $Y$ by
$$
SS = \sum_{i = 1}^{k} \sum_{j = 1}^{n _ {i}} \left(y_{ij} - \bar{y}\right)^{2}.
$$

------------

To find $SS$, we find the sum of squared distances of each observation to the overall average (i.e., the dashed line). It seems intuitive and can be shown that the total variation SS is equal to the sum of the between-groups variation $ SS_{B} $ and the within-groups variation $ SS_{W}$.
$$
SS = SS_{B} + SS_{W}.
$$

In other words, the total variation can be attributed partly to the variation within groups and partly to the variation between groups. $ SS_{B}$ is interpreted as the part of total variation $SS$ that is associated with (and can be explained by) the factor variable $X$ (e.g., syndrome type). In contrast, $SS_{W}$ is regarded as the unexplained part of total variation and is regarded as random.

In our example, if `Tetrahydrocortisone` does not depend on the type of syndrome, we expect the group-specific averages to be the same. That is, we expect the solid lines to lie on the dashed line and any observed variation of solid lines around the dashed line to be due to chance alone. On the other hand, if there is a substantial difference in `Tetrahydrocortisone` depending on the type of syndrome, then we would expect the variation between groups to be large. We examine the amount of between-groups variation relative to the variation within groups (which occurs randomly).

Let us denote the overall population mean of $Y$ as $\mu$ and group-specific population means as $\mu_{1}, \ldots, \mu_{4} $. Then we can express the null hypothesis of no difference in means between the groups as

$$
H_{0}: \mu_{1} = \mu_{2} = \mu_{3} = \mu_{4} = \mu.
$$

That is, if we had data for the whole population, the small solid lines would lie on the dashed line. The alternative hypothesis $H_{A}$ is that at least one of the group means $\mu_{i}$ is different from the overall mean $\mu$.

------------------------

>The process of evaluating hypotheses regarding the group means of multiple populations is called the **Analysis of Variance (ANOVA)**. Since we are only considering one factor only, this method is specifically called **one-way ANOVA**. The test statistic for examining the null hypothesis is called $F$-statistic (more specifically, ANOVA $F$-statistic) and is defined as
$$
F = \frac {S S _ {B} / (k - 1)}{S S _ {W} / (n - k)},
$$
where $n$ is the total sample size, and $k$ is the number of groups. The numerator is called the **mean square for groups**, and the denominator is called the **mean square error** (MSE).
Note that the above test statistic is based on comparing the variation between groups (which is explained by the factor) and the variation within groups (which is unexplained and random). When the group means are substantially different, and their variation is relatively large compared to the random variations within groups, the value of the $F$ statistic becomes large.
We denote the observed value of the $F$-statistic as $f$. If the null hypothesis is true, then the test statistic $F$ has an $F$-distribution.

---------------

The $F$-distribution, which is a continuous probability distribution, is very important for hypothesis testing. It is specified by two parameters, $df_{1}$ and $df_{2}$ , and is denoted as $F(df_{1}, df_{2})$ . We refer to $df_{1}$ and $df_{2}$ as the _numerator degrees of freedom_ and _denominator degrees of freedom_, respectively. Both parameters must be positive. Figure 9.3 shows the pdf of $F$-distribution for different values of $df_{1}$ and $df_{2}$.

![__](https://biologicslab.co/STA1403/images/chapter_09/chapter_09A_image02A.png)

>**Fig. 9.3** Comparing the plots of the probability density function for an F-distribution with various degrees of freedom. The _solid line_ represents the pdf of $ F(1,1)$, the _dashed line_ represents the pdf of $F(2,5)$, and the _dotted line_ represents the pdf of $F(10,20)$.

-------------------------

>For the one-way ANOVA, the $F$-statistic has $F(df_{1}=k-1, df_{2}=n-k)$ distribution under the null hypothesis (i.e., assuming that the null hypothesis is true). Here, $df_{1}=k-1$, which is the number of groups minus 1, is called the numerator degrees of freedom, and $df_{2}=n-k$, which is the sample size minus the number of groups, is called the denominator degrees of freedom. The underlying assumption here is that the observations in each group are IID (e.g., obtained through SRS) and have a normal distribution. The results are not sensitive to the normality assumption as long as the sample sizes are large enough for the CLT to hold.
Additionally, the underlying assumption of the ANOVA method discussed here is that all groups have the same population variance, $\sigma^{2}$, which is unknown.

--------------------

For the above example, the degrees of freedom parameters are $df_{1}=4-1=3 $ and $ df_{2}=27-4=23$. When the group means are very different from each other, the between-groups variation $ SS_{B} $ is high. As a result, the $F$-statistic is large. Therefore, large values of the $F$-statistic are considered as extreme if the null hypothesis is true. Therefore, large values of $F$ provide strong evidence _against the null hypothesis_.

![__](https://biologicslab.co/STA1403/images/chapter_09/chapter_09A_image03A.png)

>**Fig. 9.4** The density plot of $F(3,23)$-distribution. This is the distribution of $F$-statistic for the Cushings data assuming that the null hypothesis is true. The observed value of the test statistic is $f = 3.2$, and the corresponding $p$-value is shown as the shaded area above 3.2.

To find the observed significance level $p_{obs}$, we find the probability of values as or more extreme than the observed value of the test statistic, $f$. For this, we calculate the upper tail probability of $f$ based on an $F(df_{1}=k-1, df_{2}=n-k) $ distribution. For the above example, the observed value of the test statistic is $f = 3.2$. Therefore, the $p$-value is $p_{obs}=P(F\geq3.2)$, which is shown as the shaded area in Fig. 9.4.

Using Python, we can directly perform the Analysis of Variance (ANOVA) as shown in Example 2.

## Example 2: Analysis of Variance (ANOVA)

The code in the cell below performs an Analysis of Variance (ANOVA) on the `Cushings` dataset. Specifically, the code uses ANOVA to test whether the urinary excretion rate of the metabolite `Pregnanetriol` is different in patients with different types of adrenocortical tumors ($a$, $b$, $c$ or $u$).


In [ ]:
# @title Example 2: Analysis of Variance (ANOVA)

import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

# ── INPUT SECTION ─────────────────────────────────────────────────────────────
anova_factor_1  = "PregN"
factor_1_name   = "Pregnanetriol"
anova_factor_2  = "Type"
# ──────────────────────────────────────────────────────────────────────────────

# Group by 'anova_factor_1` and `anova_factor_2`
summary_table = (
    df_cushings.groupby(anova_factor_2)[anova_factor_1]
    .agg(mean="mean", sd="std", n="count")
)

print(f" **** Summary Statistics for {factor_1_name} ****")
print(summary_table)
print("\n")

# 1. Define the R-style significance function
def get_sig_stars(p_value):
    if pd.isna(p_value):
        return ""
    if p_value < 0.001:
        return "***"
    elif p_value < 0.01:
        return "**"
    elif p_value < 0.05:
        return "*"
    elif p_value < 0.1:
        return "."
    else:
        return " "
# 2. Combine variables into a single string variable
formula = f"{anova_factor_1} ~ {anova_factor_2}"
print(f"ols() formula = {formula}\n")

# 3. Perform One-Way ANOVA
model = ols(formula, data=df_cushings).fit()

# 4. Generate ANOVA table
anova_table = sm.stats.anova_lm(model, typ=2)

# 5. Add the significance stars column based on 'PR(>F)'
anova_table['Signif'] = anova_table['PR(>F)'].apply(get_sig_stars)

# 6. Print the updated table
print(f" **** ANOVA Table ****")
print(anova_table)
print("\nSignif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1")

If the code is correct, you should see the following output:
```text
 **** Summary Statistics for Pregnanetriol ****
      mean        sd   n
Type                    
a     2.44  4.565961   6
b     1.12  1.003106  10
c     5.50  2.536730   5
u     1.20  1.887856   6


ols() formula = PregN ~ Type

 **** ANOVA Table ****
              sum_sq    df         F    PR(>F) Signif
Type       72.411467   3.0  3.539263  0.030507      *
Residual  156.856000  23.0       NaN       NaN       

Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
```

The `Summary Statistics Table` at the top of the output shows the group-specific means, the group-specific standard deviations and the number of observations ($n$) in each group ($n_i$) for `Pregnanetriol`.

Below is the `ANOVA Table` for `Pregnanetriol`. The first row of the ANOVA table is for the group variable (`Type`) and shows the explained part of the total variation (i.e., between groups). The last row (`Residuals`) shows the unexplained part (i.e., random variations within groups) of the total variation in the data . The first column shows the degrees of freedom (`df`), which are $k − 1 = 3$ and $n − k = 23$, respectively. The values of the second column, labeled `sum sq`, are the between-groups and within-groups variations: $SS_B = 72.4115$ and $SS_W = 156.856$. The observed value of $F$-statistic is $f = 3.54$ given under the column labeled $F$ value. The resulting $p$-value is then $0.031$. Since the $p$-value is less than $0.05$, the $F$-statistic is statistically significant.

## **Exercise 2: Analysis of Variance (ANOVA)**

In the cell below, write the code to perform an analysis of variance (ANOVA) on the urinary metabolite `Tetrahydrocortisone` by syndrome type.


**Code Hints:**

1. Copy-and-paste Example 2 into the cell below
2. Change the variables in the INPUT SECTION to read as shown in the following code chunk:

```Python
# ── INPUT SECTION ─────────────────────────────────────────────────────────────
anova_factor_1  = "TCort"
factor_1_name   = "Tetrahydrocortisone"
anova_factor_2  = "Type"
# ──────────────────────────────────────────────────────────────────────────────
```

In [ ]:
# @title Exercise 2: Analysis of Variance (ANOVA)



If the code is correct, you should see the following output:
```text
 **** Summary Statistics for Tetrahydrocortisone ****
           mean         sd   n
Type                          
a      2.966667   0.924482   6
b      8.180000   3.789107  10
c     19.720000  19.238815   5
u     14.016667  10.095824   6


ols() formula = TCort ~ Type


 **** ANOVA Table ****
               sum_sq    df         F    PR(>F) Signif
Type       893.521000   3.0  3.225739  0.041218      *
Residual  2123.645667  23.0       NaN       NaN       

Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1
```

The `Summary Statistics Table` at the top of the output shows the group-specific means, the group-specific standard deviations and the number of observations ($n$) in each group ($n_i$) for `Tetrahydrocortisone`.

Below is the `ANOVA Table` for `Tetrahydrocortisone`. The first row of the ANOVA table is for the group variable (`Type`) and shows the explained part of the total variation (i.e., between groups). The last row (`Residuals`) shows the unexplained part (i.e., random variations within groups) of the total variation in the data . The first column shows the degrees of freedom (`df`), which are $k − 1 = 3$ and $n − k = 23$, respectively. The values of the second column, labeled `sum sq`, are the between-groups and within-groups variations: $SS_B = 893.521$ and $SS_W = 92.3324$. The observed value of $F$-statistic is $f = 3.223$ given under the column labeled $F$ value. The resulting $p$-value is then $0.041$. Since the $p$-value is less than $0.05$, the $F$-statistic is statistically significant.

In Fig. 9.7, we showed the dot plot of Tetrahydrocortisone by syndrome type. Alternatively, we can show this data as a _plot of means_.

![__](https://biologicslab.co/STA1403/images/chapter_09/chapter_09A_image04A.png)

>**Fig. 9.7** Plot of means for Tetrahydrocortisone by syndrome type. The points show the location of the sample mean for the corresponding syndrome type. The bars show the 95% confidence intervals around the sample means

Example 3 show how we can use Python to create a plot of means similar to Fig. 9.7.

## Example 3: Plot of Means

The code in the cell below shows how to generate a plot of means similar to the one shown in Fig. 9.7. for `Tetrahydrocortisone` excretion for each tumor cell type.

In [ ]:
# @title Example 3: Plot of Means

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ── INPUT SECTION ─────────────────────────────────────────────────────────────
anova_factor_1  = "TCort"
factor_name     = "Tetrahydrocortisone"
anova_factor_2  = "Type"
categories = ['a', 'b', 'c', 'u']
# ──────────────────────────────────────────────────────────────────────────────

# 1. Calculate mean and 95% CI (approx 1.96 * SEM)
# Group by Type and calculate mean and standard error of the mean (SEM)
stats = df_cushings.groupby(anova_factor_2)[anova_factor_1].agg(['mean', 'sem']).reset_index()
stats['ci'] = stats['sem'] * 1.96 * 2

# Ensure the order of Types is correct
stats[anova_factor_2] = pd.Categorical(stats[anova_factor_2], categories=categories, ordered=True)
stats = stats.sort_values(anova_factor_2)

# 2. Plotting
plt.figure(figsize=(6, 5), facecolor='white')
plt.rcParams['axes.facecolor'] = 'white'

# Plot means as a line graph with black circle markers
plt.plot(stats[anova_factor_2], stats['mean'], color='black', marker='o', markersize=8, linestyle='solid', linewidth=1.0)

# 3. Add vertical dashed error bars
plt.errorbar(stats[anova_factor_2], stats['mean'], yerr=stats['ci'], fmt='none',
             ecolor='black', capsize=4, linestyle='dotted', linewidth=0.4)

# 4. Labels and Title
plt.title('Plot of Means', fontsize=14)
plt.xlabel('Type', fontsize=12)
plt.ylabel(factor_name, fontsize=12)

# 5. Minimalist style
plt.grid(False)
plt.gca().spines['top'].set_visible(True)
plt.gca().spines['right'].set_visible(True)
plt.gca().tick_params(axis='both', colors='black')

plt.tight_layout()


If the code is correct, you should see the following output:

![__](https://biologicslab.co/STA1403/images/chapter_09/chapter_09A_image05A.png)

Based on this graph, the type “a” syndrome has the lowest sample mean. The sample means increase from type “a” to type “c”, and then it slightly drops for type “u”.

## **Exercise 3: Plot of Means**

In the cell below, generate a plot of means for the metabolite `Pregnanetriol` for each tumor type.

**Code Hints:**

**Code Hints:**

1. Copy-and-paste Example 3 into the cell below
2. Change the variables in the INPUT SECTION to read as shown in the following code chunk:"

```Python
# ── INPUT SECTION ─────────────────────────────────────────────────────────────
anova_factor_1  = "PregN"
factor_name     = "Pregnanetriol"
anova_factor_2  = "Type"
categories = ['a', 'b', 'c', 'u']
# ──────────────────────────────────────────────────────────────────────────────
```

In [ ]:
# @title Exercise 3: Plot of Means



If the code is correct, you should see the following output:

![__](https://biologicslab.co/STA1403/images/chapter_09/chapter_09A_image06A.png)

# **Electronic Submission**

When you run the code in the cell below, it will grade your Colab notebook and tell you your pending grade as it currently stands. You will be given the choice to either submit your notebook for final grading or the option to continue your work on one (or more) Exercises.

Grant Access to your Colab Secrets if you are asked to do so.

In [ ]:
# @title Electronic Submission

import urllib.request
import ssl
import time

url = "https://biologicslab.co/STA1403/backend_code/validate.py?v=" + str(time.time())

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

req = urllib.request.Request(
    url,
    headers={
        "Cache-Control": "no-cache, no-store, must-revalidate",
        "Pragma": "no-cache",
        "Expires": "0"
    }
)

with urllib.request.urlopen(req, context=ctx) as r:
    exec(r.read().decode("utf-8"))

main()